In [ ]:
# SimpleDirectoryReader is dynamic, detects file type and uses appropriate reader
from llama_index.core import SimpleDirectoryReader, Document, VectorStoreIndex, Settings, PromptTemplate, StorageContext, KnowledgeGraphIndex
from llama_index.core.utilities.sql_wrapper import SQLDatabase
from llama_index.core.query_engine import NLSQLTableQueryEngine, KnowledgeGraphQueryEngine
from llama_index.core.workflow import Workflow, StartEvent, StopEvent, step, Context, Event
from llama_index.core.retrievers import SQLRetriever
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.openai import OpenAI
from llama_parse import LlamaParse

from transformers import AutoTokenizer

import pandas as pd, re, ast, textwrap
from sqlalchemy import create_engine

import openai
from neo4j import GraphDatabase
from llama_index.graph_stores.neo4j import Neo4jGraphStore

from dotenv import load_dotenv, find_dotenv
import torch
import os
import re
import json

# 1. Automatically find the .env file by searching up the directory tree.
#    This is the key step. It makes the code work from any subdirectory.
dotenv_path = find_dotenv()

# 2. Load the .env file from the path that was found.
load_dotenv(dotenv_path=dotenv_path)

# 3. Get the project root from the directory where the .env file was found.
project_root = os.path.dirname(dotenv_path)

# 4. Get the relative directory name from the environment variable
relative_data_dir = os.getenv("VECTOR_DATASET_DIR")

# 5. *** THIS IS THE CRITICAL FIX ***
#    Create the full, absolute path by joining the project root with the relative name.
data_directory = os.path.join(project_root, relative_data_dir)

hf_token = os.getenv("HUGGINGFACE_TOKEN")
# data_directory = os.getenv("VECTOR_DATASET_DIR")
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")


print(f"✅ Project root automatically determined as: {project_root}")
print(f"✅ .env file loaded from: {dotenv_path}")
print(f"📁 Data directory set to: {data_directory}")

Connection to local Neo4j Desktop database successful!


## Vector/Graph Store Data Ingestion

In [ ]:
# Check if directory exists
if not data_directory or not os.path.isdir(data_directory):
    raise ValueError(
        f"The path '{data_directory}' is not a valid directory. "
        "Please check that the VECTOR_DATASET_DIR variable is set correctly in your .env file "
        "and that the directory actually exists."
    )

# Initialize LlamaParse with your API key
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
if not llama_cloud_api_key:
    raise ValueError("LLAMA_CLOUD_API_KEY not found in your .env file. Please get a key from https://cloud.llamaindex.ai")

parser = LlamaParse(
    api_key=llama_cloud_api_key,
    result_type="markdown",
    verbose=True
)

# Separate the file paths based on their type (PDF vs. other)
pdf_filepaths = []
other_filepaths = []
for filename in os.listdir(data_directory):
    file_path = os.path.join(data_directory, filename)
    if os.path.isfile(file_path):
        if filename.lower().endswith('.pdf'):
            pdf_filepaths.append(file_path)
        else:
            other_filepaths.append(file_path)

print(f"--- Found {len(pdf_filepaths)} PDF(s) and {len(other_filepaths)} other file(s) to process. ---")

# Process the files in batches
all_documents = []

# Process all PDFs in a single batch call to LlamaParse
if pdf_filepaths:
    print("\n- Parsing PDF files with LlamaParse...")
    try:
        # Calling parser.load_data() with a LIST of files is the correct way
        pdf_docs = parser.load_data(pdf_filepaths)
        all_documents.extend(pdf_docs)
        print(f"  -> Successfully parsed {len(pdf_filepaths)} PDF file(s).")
    except Exception as e:
        print(f"  -> FAILED to parse PDFs with LlamaParse. Error: {e}")

# Process all other files in a single batch call to SimpleDirectoryReader
if other_filepaths:
    print("\n- Parsing other files with SimpleDirectoryReader...")
    try:
        other_docs = SimpleDirectoryReader(input_files=other_filepaths).load_data()
        all_documents.extend(other_docs)
        print(f"  -> Successfully parsed {len(other_filepaths)} other file(s).")
    except Exception as e:
        print(f"  -> FAILED to parse other files. Error: {e}")

# The 'documents' variable should now contain all chunks from all parsed files
documents = all_documents
print(f"\n--- Ingestion complete ---")
print(f"Successfully loaded and chunked a total of {len(documents)} document(s) from all files in '{data_directory}'.")


--- Found 1 PDF(s) and 0 other file(s) to process. ---

- Parsing PDF files with LlamaParse...


Parsing files:   0%|          | 0/1 [00:00<?, ?it/s]

Started parsing the file under job_id 2c7e14dc-3b6a-4909-aaf8-01efd7c006fe


Parsing files: 100%|██████████| 1/1 [00:46<00:00, 46.83s/it]

  -> Successfully parsed 1 PDF file(s).

--- Ingestion complete ---
Successfully loaded and chunked a total of 2 document(s) from all files in 'Graph_Dataset'.


In [ ]:
# Create a node parser with overlapping chunks
node_parser = SentenceSplitter(
    chunk_size=512, 
    chunk_overlap=20
)

# Get nodes from the LlamaParse documents
nodes = node_parser.get_nodes_from_documents(documents)

print(f"📄 Split documents into {len(nodes)} nodes using SentenceSplitter.")


📄 Split documents into 3 nodes using SentenceSplitter.


## Llama 3.1 8B Instruct

In [ ]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Initialize the tokenizer to get the token ID for our stop sequence
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
# The semicolon is our desired stop character. Get its token ID.
semicolon_token_id = tokenizer.convert_tokens_to_ids(";")

# Now, initialize the LLM with the correct stop condition
llm = HuggingFaceLLM(
    model_name=model_name,
    tokenizer_name=model_name,
    device_map="auto",
    model_kwargs={"token": hf_token, "torch_dtype": torch.bfloat16},
    # Use 'eos_token_id' which is the correct parameter for this purpose
    generate_kwargs={
        "temperature": 0.1,
        "do_sample": True,
        # This tells the model to stop generating as soon as it outputs a semicolon
        "eos_token_id": semicolon_token_id,
    }
)

print("HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.


## OpenAI gpt model (Entity & Relationship Extraction for Graph datastore)

In [ ]:
openai_llm = OpenAI(
    api_key=openai_api_key,
    model="gpt-4o",
    temperature=0.0,
    # Uncomment to set a timeout
    # timeout=120.0,
)

print("OpenAI LLM initialized successfully.")

OpenAI LLM initialized successfully.


## Entity & Relationship Extraction with gpt

In [ ]:
kg_extraction_prompt_str = """
You are an expert data extraction algorithm. Your task is to extract a knowledge graph from the provided text.
The knowledge graph should consist of nodes and relationships.

**Instructions:**
1.  **Nodes:** Identify all relevant entities and assign them one of the following labels:
    - {allowed_nodes}

2.  **Relationships:** Identify the relationships between these entities. The relationship triplets MUST follow this format: `(Head Entity, RELATIONSHIP_TYPE, Tail Entity)`.
    - The relationship type MUST be one of the following:
    - {allowed_relationships_str}

3.  **Output Format:** Your response MUST be ONLY a single, valid JSON list of triplets, where each triplet is a list of three strings: `["head_entity", "relationship_type", "tail_entity"]`.
    - Do not include any explanations, introductory text, or markdown formatting.
    - If no relationships are found, return an empty list `[]`.

**Example:**
Text: "HTX signed an agreement with Microsoft in Redmond."
JSON Output:
[["HTX", "SIGNED_AGREEMENT_WITH", "Microsoft"], ["Microsoft", "LOCATED_IN", "Redmond"]]

---
**Text to Analyze:**
{text}
---
**JSON Output:**
"""

# Define your schema
allowed_nodes = [
    "Organization", "Person", "Location", "Project", "Grouping",
    "PartnerCategory", "Concept", "Value", "Description", "Technology"
]

# Use the three-tuple format to define relationship schema
allowed_relationships = [
    ("Organization", "HAS_GROUPING", "Grouping"),
    ("Grouping", "CONTAINS_CATEGORY", "PartnerCategory"),
    ("PartnerCategory", "INCLUDES_PARTNER", "Organization"),
    # ... add all other relationship tuples from your manual graph here
]

# Format the allowed relationships for the prompt
allowed_relationships_str = "\n".join([f"- {rel[1]} (Connects {rel[0]} to {rel[2]})" for rel in allowed_relationships])

# Create the LlamaIndex PromptTemplate
kg_extraction_prompt = PromptTemplate(
    kg_extraction_prompt_str,
    prompt_type="knowledge_graph_extraction",
)

print("✅ Schema-enforcing prompt for LlamaIndex is ready.")

✅ Schema-enforcing prompt for LlamaIndex is ready.


## Graph Helper Functions

In [ ]:
def filter_triplets(triplets: list, allowed_nodes: list, allowed_relationships: list) -> list:
    """
    Filters extracted triplets to ensure they conform to the predefined schema.
    This acts as the "strict_mode=True" for LlamaIndex.
    """
    # This function is not needed for the LlamaIndex KnowledgeGraphIndex as it handles this internally
    # when you provide the allowed_nodes and allowed_relationships directly.
    # However, it's a good practice to have it for custom RAG pipelines.
    # For this specific implementation, we will rely on the index's internal validation.
    # Kept here for educational purposes.
    return triplets


## Manual Ingestion into Neo4j function

In [ ]:
# Helper function to create a relationship between two nodes.
def create_manual_relationship(driver, subject_label, subject_name, rel_type, object_label, object_name):
    """
    Creates two nodes and a relationship between them using explicit data.
    It uses MERGE to avoid creating duplicate nodes or relationships.
    """
    with driver.session() as session:
        # This Cypher query is robust. It finds or creates the subject and object nodes,
        # then finds or creates the relationship between them.
        query = f"""
        MERGE (s:{subject_label} {{name: $subject_name}})
        MERGE (o:{object_label} {{name: $object_name}})
        MERGE (s)-[:`{rel_type}`]->(o)
        """
        session.run(query, subject_name=subject_name, object_name=object_name)
    print(f"  - Created/Verified: ({subject_name})-[{rel_type}]->({object_name})")

## Manual Creation of Knowledge Graph

In [ ]:
# Clear DB before ingesting additional relations
driver = GraphDatabase.driver(uri, auth=(username, password))
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("Database cleared.")

# Create the root HTX node and partner category
create_manual_relationship(
    driver,
    subject_label="Root", subject_name="HTX",
    rel_type="HAS_GROUPING",
    object_label="Grouping", object_name="Partners"
)

all_partners = {
    "Strategic Partners for Innovation": ["accenture", "Microsoft", "IDEMIA", "NCS", "Singtel", "Thales", "ENSIGN", "KLASS", "NEC", "PCSS", "ROHDE&SCHWARZ", "novel", "SHIMADZU", "ST Engineering", "SEKISUI", "veredus"],
    "Academic Partners": ["MIT", "A*STAR", "Fraunhofer", "IEEE CSAIL", "NTU", "NUS", "SIT", "SUTD", "Singapore Polytechnic"],
    "Local Government Agencies": ["CSA", "DSO", "DSTA", "GOVTECH", "HSA", "IMDA", "LTA", "MINDEF", "MHA", "MOM", "NEA", "NATIONAL PARKS"],
    "Foreign Government Agencies": ["Australian Border Force", "AFP", "Ministry of National Security", "French Ministry of the Interior", "Dutch Ministry of Justice and Security", "US Department of Homeland Security", "KOREAN NATIONAL POLICE AGENCY"],
    "Other Key Industry Partners": ["aws", "CERTIS", "CISCO", "SAMSUNG", "SANS", "sats", "SOSA"]
}

# C. Loop through the data to build the graph
for category, partners in all_partners.items():
    # Link the category to the main 'Partners' grouping
    create_manual_relationship(
        driver,
        subject_label="Grouping", subject_name="Partners",
        rel_type="CONTAINS_CATEGORY",
        object_label="PartnerCategory", object_name=category
    )
    
    # Link each partner to its category
    for partner in partners:
        create_manual_relationship(
            driver,
            subject_label="PartnerCategory", subject_name=category,
            rel_type="INCLUDES_PARTNER",
            object_label="Organization", object_name=partner
        )

# Create the root HTX node and who we are category
create_manual_relationship(
    driver,
    subject_label="Root", subject_name="HTX",
    rel_type="HAS_GROUPING",
    object_label="Grouping", object_name="Who We Are"
)

# Who We Are -> Our Values
create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Who We Are",
    rel_type="CONTAINS_VALUES",
    object_label="Values", object_name="Our Values"
)

create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Who We Are",
    rel_type="CONTAINS_CULTURE",
    object_label="Concept", object_name="Our Culture & People"
)

create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Who We Are",
    rel_type="CONTAINS_COMMITMENT",
    object_label="Concept", object_name="Our Commitment to Sustainability"
)

create_manual_relationship(
    driver,
    subject_label="Grouping", subject_name="Who We Are",
    rel_type="CONTAINS_GOALS",
    object_label="Concept", object_name="Our Goals"
)

# Our Values -> Mission
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Values",
    rel_type="CONTAINS_VALUE",
    object_label="Value", object_name="Mission"
)

# Our Values -> Mission
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Values",
    rel_type="CONTAINS_VALUE",
    object_label="Value", object_name="Teamwork"
)

# Our Values -> Mission
create_manual_relationship(
    driver,
    subject_label="Value", subject_name="Teamwork",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="We work together to make the extraordinary happen"
)

# Our Values -> Mission
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Values",
    rel_type="CONTAINS_VALUE",
    object_label="Value", object_name="Empathy"
)

# Our Values -> Mission
create_manual_relationship(
    driver,
    subject_label="Value", subject_name="Empathy",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="We appreciate and care for one another, and celebrate our achievements together"
)

# Our Values -> Mission
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Values",
    rel_type="CONTAINS_VALUE",
    object_label="Value", object_name="Exuberance"
)

# Our Values -> Mission
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Values",
    rel_type="CONTAINS_VALUE",
    object_label="Value", object_name="Foresight"
)

# Our Values -> Mission
create_manual_relationship(
    driver,
    subject_label="Concept", subject_name="Our Values",
    rel_type="CONTAINS_VALUE",
    object_label="Value", object_name="Innovation"
)

# Our Values -> Mission
create_manual_relationship(
    driver,
    subject_label="Value", subject_name="Mission",
    rel_type="HAS_DESCRIPTION",
    object_label="Description", object_name="We are the Home Team’s Force Multiplier"
)


print("\n--- Manual knowledge graph creation complete. ---")

driver.close()

Database cleared.
  - Created/Verified: (HTX)-[HAS_GROUPING]->(Partners)
  - Created/Verified: (Partners)-[CONTAINS_CATEGORY]->(Strategic Partners for Innovation)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(accenture)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(Microsoft)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(IDEMIA)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(NCS)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(Singtel)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(Thales)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(ENSIGN)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(KLASS)
  - Created/Verified: (Strategic Partners for Innovation)-[INCLUDES_PARTNER]->(NEC)
  - Created/Verified: (Strategic Partners for Innovati

## Graph Query Engine

In [ ]:
graph_store = Neo4jGraphStore(username, password, uri)

storage_context = StorageContext.from_defaults(graph_store=graph_store)

# 2. Build the KnowledgeGraphIndex with your schema
# The index constructor will extract triplets using your custom prompt and store them.
kg_index = KnowledgeGraphIndex(
    nodes,  # Use the nodes created by SentenceSplitter
    storage_context=storage_context,
    kg_extraction_prompt_template=kg_extraction_prompt,
    llm=openai_llm,  # Use your powerful GPT-4o for extraction
    max_triplets_per_chunk=15,
    include_embeddings=True,  # Recommended for hybrid search later
    # Pass the schema for LlamaIndex's internal validation (acts like strict_mode)
    allowed_kg_nodes=allowed_nodes,
    allowed_kg_relationships=[r[1] for r in allowed_relationships]
)

print("✅ KnowledgeGraphIndex built and data ingested into Neo4j.")

# Prompt template to generate cypher query
DEFAULT_KG_QUERY_SYNTHESIS_TMPL = (
"You are an expert Cypher query generator. Your sole task is to generate a single, "
"syntactically correct Cypher query to answer the user's question based on the provided graph schema. "
"Do not provide any explanations, introductory text, or markdown formatting. "
"Your response MUST be ONLY the raw Cypher query and nothing else. "
"Start your response directly with a Cypher keyword like 'MATCH' or 'OPTIONAL MATCH'.\n\n"
"Schema:\n"
"---------------------\n"
"{schema}\n"
"---------------------\n"
"User's Question: {query_str}\n"
"Cypher Query:"
)

# Prompt template to generate response
DEFAULT_RESPONSE_SYNTHESIS_TMPL = (
    "You are a helpful assistant. You have been provided with the results of a Cypher query "
    "from a knowledge graph and the original user question. "
    "Synthesize a conversational answer based on the provided information. "
    "Do not mention Cypher or the knowledge graph in your response.\n"
    "User question: {query_str}\n"
    "Query results: {context_str}\n"
    "Answer: "
)

kg_query_synthesis_prompt = PromptTemplate(DEFAULT_KG_QUERY_SYNTHESIS_TMPL)
kg_response_answer_prompt = PromptTemplate(DEFAULT_RESPONSE_SYNTHESIS_TMPL)


# Initialize the query engine with the storage_context
graph_query_engine = KnowledgeGraphQueryEngine(
    storage_context=storage_context,
    graph_query_synthesis_prompt=kg_query_synthesis_prompt,
    graph_response_answer_prompt=kg_response_answer_prompt,
    verbose=True # Check Cypher Query
)

print("Knowledge graph query engine is ready.")

## Query Graph Datastore

In [ ]:
# Define question for graph
query_text_graph = "How much was lost to job scam?"

# Query the engine
graph_response = graph_query_engine.query(query_text_graph)

# Print the response
print(str(graph_response))